In [ ]:
import pandas as pd
import pathlib
import sys
import joblib
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import random
import colorsys


script_directory = pathlib.Path("../utils/").resolve()
sys.path.insert(0, str(script_directory))
from data_loader import load_model_data, load_train_test_data

In [ ]:
def generate_random_palette(num_colors, seed=12):
    # Generate random colors with varied lightness and saturation
    random.seed(seed)
    
    colors = []
    
    for _ in range(num_colors):
        h = random.random()  # Random hue between 0 and 1
        l = random.uniform(0.2, 0.8)  # Lightness between 0.3 and 0.9 for contrast
        s = random.uniform(0.5, 1.0)  # Saturation between 0.6 and 1.0 for vivid colors
        color = colorsys.hls_to_rgb(h, l, s)  # Convert HLS to RGB color
        colors.append(color)
    
    return colors

In [2]:
# Load dependency data
data_directory = pathlib.Path("../0.data-download/data").resolve()
dependency_file = pathlib.Path(f"{data_directory}/CRISPRGeneEffect.parquet").resolve()
gene_dict_file = pathlib.Path(f"{data_directory}/CRISPR_gene_dictionary.parquet").resolve()
dependency_df, gene_dict_df= load_model_data(dependency_file, gene_dict_file)
dependency_df = dependency_df.set_index("ModelID")

(1150, 18444)


In [3]:
cancer_type_input_file = pathlib.Path("../0.data-download/data/Model.parquet")
cancer_type_df = pd.read_parquet(cancer_type_input_file)

In [4]:
combined_df = dependency_df.merge(
    cancer_type_df[["ModelID", "OncotreePrimaryDisease"]], 
    on="ModelID", 
    how="left"
)

In [5]:
combined_df.head()

,ModelID,MKLN1,PNISR,RRP12,CCNC,RAPH1,IARS1,ATXN2L,TRRAP,IBA57,...,ACTR8,NOC3L,FITM2,PSMD4,WIZ,ZCRB1,ATP6V1B2,ATP5MJ,UVRAG,OncotreePrimaryDisease
0,ACH-000001,-0.171664,-0.269956,-1.468408,0.200540,0.113552,-2.437427,-0.248702,-1.170411,-0.318523,...,-0.251498,-0.476365,0.181363,-1.941100,-0.062205,-0.188517,-0.722523,0.024160,-0.025479,Ovarian Epithelial Tumor
1,ACH-000004,0.228512,-0.397136,-1.409154,-0.397439,-0.123994,-1.725931,-0.618666,-0.932108,0.082256,...,-0.704149,-0.478794,-0.145720,-1.303979,-0.278718,-0.679133,-2.612545,-0.043210,-0.365718,Acute Myeloid Leukemia
2,ACH-000005,0.070309,-0.208097,-0.782796,-0.263901,-0.050738,-1.951187,-0.093201,-1.187058,0.007900,...,-0.175668,-0.986281,-0.020722,-1.336103,-0.031358,-0.560265,-2.434407,0.212494,-0.278459,Acute Myeloid Leukemia
3,ACH-000007,0.053490,-0.709150,-1.545416,-0.116803,-0.089551,-1.566640,-0.687175,-1.262199,-0.219823,...,-0.724938,-0.408651,-0.139004,-1.894634,-0.132096,-0.653545,-1.926781,0.065308,-0.156535,Colorectal Adenocarcinoma
4,ACH-000009,-0.153007,-0.310527,-1.297714,-0.100032,-0.394965,-1.853329,-0.362086,-1.465030,-0.098795,...,-0.396564,-0.585670,-0.085462,-2.150424,-0.038581,-0.495035,-1.449962,0.092776,-0.161901,Colorectal Adenocarcinoma


In [6]:
gene_cols = combined_df.columns.drop("ModelID")
gene_cols = gene_cols.drop("OncotreePrimaryDisease")

In [7]:
features = combined_df[gene_cols]

In [8]:
features

,MKLN1,PNISR,RRP12,CCNC,RAPH1,IARS1,ATXN2L,TRRAP,IBA57,EXOC5,...,CDKN2C,ACTR8,NOC3L,FITM2,PSMD4,WIZ,ZCRB1,ATP6V1B2,ATP5MJ,UVRAG
0,-0.171664,-0.269956,-1.468408,0.200540,0.113552,-2.437427,-0.248702,-1.170411,-0.318523,-0.590925,...,0.305990,-0.251498,-0.476365,0.181363,-1.941100,-0.062205,-0.188517,-0.722523,0.024160,-0.025479
1,0.228512,-0.397136,-1.409154,-0.397439,-0.123994,-1.725931,-0.618666,-0.932108,0.082256,-0.337582,...,0.022957,-0.704149,-0.478794,-0.145720,-1.303979,-0.278718,-0.679133,-2.612545,-0.043210,-0.365718
2,0.070309,-0.208097,-0.782796,-0.263901,-0.050738,-1.951187,-0.093201,-1.187058,0.007900,-0.361119,...,0.294628,-0.175668,-0.986281,-0.020722,-1.336103,-0.031358,-0.560265,-2.434407,0.212494,-0.278459
3,0.053490,-0.709150,-1.545416,-0.116803,-0.089551,-1.566640,-0.687175,-1.262199,-0.219823,-1.059255,...,0.289685,-0.724938,-0.408651,-0.139004,-1.894634,-0.132096,-0.653545,-1.926781,0.065308,-0.156535
4,-0.153007,-0.310527,-1.297714,-0.100032,-0.394965,-1.853329,-0.362086,-1.465030,-0.098795,-0.607368,...,0.278736,-0.396564,-0.585670,-0.085462,-2.150424,-0.038581,-0.495035,-1.449962,0.092776,-0.161901
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,-0.043628,-0.367097,-1.181241,-0.155239,-0.106159,-1.472932,-0.179256,-1.244378,-0.422217,-0.451605,...,0.194939,-0.657255,-0.456921,0.079450,-2.070201,-0.045493,-0.674658,-2.488354,0.029193,-0.334027
1146,-0.186123,-0.308102,-1.494330,-0.595756,0.106747,-2.012007,-0.075488,-1.339509,-0.132484,-0.610391,...,0.147780,-0.448993,-0.589840,-0.048516,-2.279843,-0.446167,-0.847155,-2.100223,-0.022724,-0.249053
1147,-0.119882,-0.559231,-1.947235,-0.733961,-0.046904,-1.905367,-0.098661,-1.362585,-0.334204,-0.659782,...,0.314440,-0.548188,-0.664303,-0.365153,-1.723337,-0.199305,-0.403933,-2.108371,0.024270,-0.335437
1148,-0.010826,-0.934568,-1.149496,0.122469,-0.047657,-2.450202,-0.458031,-0.992763,-0.215293,-0.387643,...,0.310693,-0.363694,-0.400961,-0.093000,-2.063535,-0.396032,-0.463082,-2.407353,0.124508,-0.511275


In [ ]:

# Prepare PCA input (excluding non-numeric columns)
pca_input = combined_df[gene_cols].apply(pd.to_numeric, errors="coerce")
pca = PCA(n_components=2, random_state=0)
pca_embedding = pca.fit_transform(pca_input)

# Add PCA components to the dataframe
combined_df["PCA1"] = pca_embedding[:, 0]
combined_df["PCA2"] = pca_embedding[:, 1]

# Prepare color map for each cancer type
cancer_types = combined_df["OncotreePrimaryDisease"].unique()
color_map = px.colors.qualitative.Plotly + px.colors.qualitative.Light24 + px.colors.qualitative.Dark24
highlight_color_map = {cancer: color_map[i % len(color_map)] for i, cancer in enumerate(cancer_types)}

# Create one trace per cancer type
traces = []
for cancer in cancer_types:
    df_subset = combined_df[combined_df["OncotreePrimaryDisease"] == cancer]
    trace = go.Scatter(
        x=df_subset["PCA1"],
        y=df_subset["PCA2"],
        mode='markers',
        name=cancer,
        marker=dict(size=7),
        text=[f"{cancer} | {model_id}" for model_id in df_subset["ModelID"]],
        hoverinfo='text',
    )
    traces.append(trace)

# Create dropdown buttons
dropdown_buttons = []

for i, cancer in enumerate(cancer_types):
    # Make all grey except the selected one
    visibility = [True] * len(cancer_types)
    colors = ['lightgrey'] * len(cancer_types)
    colors[i] = highlight_color_map[cancer]

    button = dict(
        method="update",
        label=cancer,
        args=[
            {"visible": visibility,
             "marker": [{'color': colors[j]} for j in range(len(cancer_types))]},
            {"title": f"PCA Highlighted: {cancer}"}
        ]
    )
    dropdown_buttons.append(button)

# Add a button to show all with normal colors
default_colors = [highlight_color_map[cancer] for cancer in cancer_types]
dropdown_buttons.insert(0, dict(
    method="update",
    label="Show All",
    args=[
        {"visible": [True]*len(cancer_types),
         "marker": [{'color': default_colors[j]} for j in range(len(cancer_types))]},
        {"title": "PCA of DepMap Single Gene Data"}
    ]
))

# Create the figure
fig = go.Figure(data=traces)

# Set default colors
for i, trace in enumerate(fig.data):
    trace.marker.color = default_colors[i]

fig.update_layout(
    title="PCA of DepMap Single Gene Data",
    xaxis_title="PCA1",
    yaxis_title="PCA2",
    updatemenus=[{
        "buttons": dropdown_buttons,
        "direction": "down",
        "showactive": True,
        "x": 1.15,
        "xanchor": "left",
        "y": 1.15,
        "yanchor": "top"
    }],
    width=1500,
    height=800
)

fig.show()
fig.write_html("achilles_pca.html")